In [ ]:
import os
import numpy as np
import pandas as pd
import supervision as sv
from supervision.metrics import MeanAveragePrecision

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
# Configuration files
model_configs = [
    {   # Bihar to Bihar
        'train': 'Train Bihar',
        'test': 'Test Bihar',
        'backbone': 'swint',
        'head': 'rhino',
        'config_file': 'configs-mine/rhino-swint-dota2config/rhino_phc_haus-4scale_swint_2xb2-36e_bihar.py',
        'checkpoint_folder': 'work_dirs/rhino_phc_haus-4scale_swint_2xb2-36e_bihar',
        'val_dir': 'data/bihar/val',
        'inf_dir': 'results-swint/train_bihar_test_bihar',
        'img_height': 640,
        'epoch': 50,
    },
    {
        # Haryana to Bihar
        'train': 'Train Haryana',
        'test': 'Test Bihar',
        'backbone': 'swint',
        'head': 'rhino',
        'config_file': 'configs-mine/rhino-swint-dota2config/rhino_phc_haus-4scale_swint_2xb2-36e_haryana.py',
        'checkpoint_folder': 'work_dirs/rhino_phc_haus-4scale_swint_2xb2-36e_haryana',
        'val_dir': 'data/haryana/val_bihar',
        'inf_dir': 'results-swint/train_haryana_test_bihar',
        'img_height': 640,
        'epoch': 50,
    },
    {
        # m0 to m0
        'train': 'Train m0',
        'test': 'Test m0',
        'backbone': 'swint',
        'head': 'rhino',
        'config_file': 'configs-mine/rhino-swint-dota2config/rhino_phc_haus-4scale_swint_2xb2-36e_m0.py',
        'checkpoint_folder': 'work_dirs/rhino_phc_haus-4scale_swint_2xb2-36e_m0',
        'val_dir': 'data/m0/val',
        'inf_dir': 'results-swint/train_m0_test_m0',
        'img_height': 640,
        'epoch': 50,
    },
    {
        # SwinIR Bihar to Bihar
        'train': 'Train SwinIR Bihar',
        'test': 'Test SwinIR Bihar',
        'backbone': 'swint',
        'head': 'rhino',
        'config_file': 'configs-mine/rhino-swint-dota2config/rhino_phc_haus-4scale_swint_2xb2-36e_swinir_bihar.py',
        'checkpoint_folder': 'work_dirs/rhino_phc_haus-4scale_swint_2xb2-36e_swinir_bihar',
        'val_dir': 'data/swinir/test_bihar_same_class_count_10_120_1000_4x',
        'inf_dir': 'results-swint/train_swinir_bihar_test_bihar',
        'img_height': 2560,
        'epoch': 50,
    },
    {
        # SwinIR Haryana to Bihar
        'train': 'Train SwinIR Haryana',
        'test': 'Test SwinIR Bihar',
        'backbone': 'swint',
        'head': 'rhino',
        'config_file': 'configs-mine/rhino-swint-dota2config/rhino_phc_haus-4scale_swint_2xb2-36e_swinir_haryana.py',
        'checkpoint_folder': 'work_dirs/rhino_phc_haus-4scale_swint_2xb2-36e_swinir_haryana',
        'val_dir': 'data/swinir/test_bihar_same_class_count_10_120_1000_4x',
        'inf_dir': 'results-swint/train_swinir_haryana_test_bihar',
        'img_height': 2560,
        'epoch': 50,
    },
]

In [ ]:
def get_image_names_from_directory(directory):
    """Extracts image names (without extension) from a directory."""
    return {file_name.replace(".txt", "") for file_name in os.listdir(directory) if file_name.endswith(".txt")}

def load_detections(annotations_path, img_names, is_gt=True):
    """Loads detections only for images that exist in both GT and Predictions."""
    sv_data = []

    for image_id in sorted(img_names):
        file_path = os.path.join(annotations_path, f"{image_id}.txt")
        if not os.path.exists(file_path):  # Ensure file exists before processing
            continue

        xyxy_list = []
        class_ids = []
        scores = []

        with open(file_path, "r") as file:
            lines = file.readlines()

        for line in lines:
            data = list(map(float, line.split()))
            class_id = int(data[0])
            polygon = np.array(data[1:9]).reshape(4, 2)  # Convert to (4,2) shape
            score = data[9] if not is_gt else 1.0  # Default confidence for GT is 1.0

            # Convert quadrilateral to bounding box (min x, min y, max x, max y)
            x_min, y_min = np.min(polygon, axis=0)
            x_max, y_max = np.max(polygon, axis=0)
            bbox = [x_min, y_min, x_max, y_max]

            # Append to lists
            xyxy_list.append(bbox)
            class_ids.append(class_id)
            scores.append(score)

        # Convert lists into a Supervision Detections object
        detections = sv.Detections(
            xyxy=np.array(xyxy_list),
            class_id=np.array(class_ids),
            confidence=np.array(scores),
            metadata={"image_id": image_id}
        )

        sv_data.append(detections)

    return sv_data

def get_class_counts(detections_list, num_classes=3):
    """Counts occurrences of each class in ground truth detections."""
    class_counts = np.zeros(num_classes)
    for detections in detections_list:
        unique, counts = np.unique(detections.class_id, return_counts=True)
        for cls, count in zip(unique, counts):
            class_counts[cls] += count
    return class_counts

In [ ]:
index = pd.MultiIndex.from_tuples([], names=["Base State", "Target State",  "Epochs"])
result_df = pd.DataFrame(columns=["CFCBK", "FCBK", "Zigzag", "Weighted mAP@50", "mAP@50:95", "mAP@50", "mAP@75", "CA mAP@50:95", "CA mAP@50", "CA mAP@75"], index=index)

In [ ]:
model_config = model_configs[4]

In [ ]:
for epoch in range(1, model_config['epoch'] + 1)[::-1]:
    # Load image names from directories
    GT_PATH = os.path.join(model_config['val_dir'], "labels")
    # print(f"GT path: {GT_PATH}")
    if not os.path.exists(GT_PATH):
        print(f"GT path {GT_PATH} does not exist.")
        break
    PREDICTIONS_PATH = os.path.join(model_config['inf_dir'], f"epoch_{epoch}_supervision_conf_0.01_nms_0.33", "annfiles")
    if not os.path.exists(PREDICTIONS_PATH):
        print(f"Predictions path {PREDICTIONS_PATH} does not exist.")
        continue
    gt_img_names = get_image_names_from_directory(GT_PATH)
    pred_img_names = get_image_names_from_directory(PREDICTIONS_PATH)
    img_names = gt_img_names.intersection(pred_img_names)
    base_state = model_config['train']
    target_state = model_config['test']

    # Load GT and Predictions
    gt_data = load_detections(GT_PATH, img_names, is_gt=True)
    pred_data = load_detections(PREDICTIONS_PATH, img_names, is_gt=False)

    # Print mAP results
    print(f"\n{model_config['train']} to {model_config['test']} (Epoch {epoch}):")
    ## mAP calculation (non-class agnostic)
    mAP_metric = MeanAveragePrecision(class_agnostic=False)
    mAP_result = mAP_metric.update(pred_data, gt_data).compute()
    matched_classes = mAP_result.matched_classes.tolist()
    # print(f"    Matched classes: {matched_classes}")
    # Extract overall mAP values
    mAP_50_95 = mAP_result.map50_95  # mAP 50:95
    mAP_50 = mAP_result.map50  # mAP 50
    mAP_75 = mAP_result.map75  # mAP 75
    print(f"    mAP 50:95: {mAP_50_95}, mAP 50: {mAP_50}, mAP 75: {mAP_75}")

    # Extract class-wise mAP
    class_wise_mAP = mAP_result.ap_per_class[:, 0].tolist()  # mAP 50:95 per class
    num_classes = 3
    final_class_wise_mAP = [0] * num_classes
    for cls, mAP in zip(matched_classes, class_wise_mAP):
        final_class_wise_mAP[cls] = mAP
    print(f"    class_wise_mAP: {final_class_wise_mAP}\n")
    # Calculate weighted mAP
    class_counts = get_class_counts(gt_data, num_classes=num_classes)
    print(f"    class_counts: {class_counts}")
    weighted_mAP_50 = np.sum(np.array(final_class_wise_mAP) * class_counts) / np.sum(class_counts)
    print(f"    Weighted mAP 50: {weighted_mAP_50}\n")

    # Compute class-agnostic mAP
    mAP_metric_agnostic = MeanAveragePrecision(class_agnostic=True)
    mAP_result_agnostic = mAP_metric_agnostic.update(pred_data, gt_data).compute()
    # Extract class-agnostic mAP values
    mAP_50_95_agnostic = mAP_result_agnostic.map50_95  # mAP 50:95
    mAP_50_agnostic = mAP_result_agnostic.map50  # mAP 50
    mAP_75_agnostic = mAP_result_agnostic.map75  # mAP 75
    print(f"    CA mAP 50:95: {mAP_50_95_agnostic}, CA mAP 50: {mAP_50_agnostic}, CA mAP 75: {mAP_75_agnostic}")

    # Update results dataframe
    result_df.loc[(base_state, target_state, epoch), :] = [f"{x:.6f}" for x in final_class_wise_mAP + [weighted_mAP_50, mAP_50_95, mAP_50, mAP_75, mAP_50_95_agnostic, mAP_50_agnostic, mAP_75_agnostic]]


In [ ]:
display(result_df)

In [ ]:
# Save the dataframe as CSV
result_df.to_csv(f"results-{model_config['backbone']}/{model_config['train']}_{model_config['test']}_{model_config['head']}_epoch_results.csv")

In [ ]:
saved_df = pd.read_csv(f"results-{model_config['backbone']}/{model_config['train']}_{model_config['test']}_{model_config['head']}_epoch_results.csv", index_col=[0, 1, 2])
# display(saved_df)

In [ ]:
ca_mAP_df = saved_df.reset_index()[["Epochs", "CA mAP@50"]].set_index("Epochs")
# display(ca_mAP_df)

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.plot(ca_mAP_df.index, ca_mAP_df["CA mAP@50"], label=f"{model_config['train']} to {model_config['test']}")
plt.xlabel("Epochs")
plt.ylabel("CA mAP@50")
plt.legend()
plt.savefig(f"results-{model_config['backbone']}/{model_config['train']}_{model_config['test']}_{model_config['head']}_epoch_results.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# # Filter to include only epochs up to 50
# ca_mAP_df = ca_mAP_df[ca_mAP_df.index <= 50]
# Sort by CA mAP@50 values
sorted_df = ca_mAP_df.sort_values(by="CA mAP@50", ascending=False)
display(sorted_df)

In [ ]:
best_epoch = sorted_df.index[0]
print(f"Best epoch: {best_epoch}")
print(f"Best CA mAP@50: {sorted_df.iloc[0, 0]}")